# EDA: Composition and Genres (Person 1)

**Owner:** Dua Raza Kazmi (2025-AI-200)

| # | Chart | Question |
|---|---|---|
| 1 | Movies vs TV series (donut) | Q2 Composition: how is the catalog split? |
| 2 | Top 15 genres (horizontal bar) | Q3 Genres: which genres dominate? |
| 8 | Genre evolution over time (stacked area) | Q3 Genres: how has the mix changed? |

**Input:** `data/raw/extracted_latest.csv` for now. After Monday's merge, switch `SOURCE` below to the database.

Each chart is saved to `figures/` as a PNG for the report.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Find the project root whether the notebook runs from notebooks/ or the root.
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

SOURCE = "csv"   # change to "db" after the merge
TMDB_CSV = ROOT / "data" / "raw" / "extracted_latest.csv"
KAGGLE_CSV = ROOT / "data" / "raw" / "netflix_titles.csv"
DB_PATH = ROOT / "netflix_analysis.db"

# Shared style. Replace with Person 4's palette.py once it exists.
PALETTE = ["#E50914", "#221F1F", "#B81D24", "#F5A623", "#4A90E2",
           "#7ED321", "#9B59B6", "#50E3C2", "#8E8E93", "#F8E71C"]
SOURCE_NOTE = "Source: TMDB API (Netflix, provider 8)"

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200,
                     "axes.titleweight": "bold", "axes.titlesize": 14})


def add_source(fig, text=SOURCE_NOTE):
    fig.text(0.99, 0.01, text, ha="right", va="bottom", fontsize=8, color="#666")


def save(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, bbox_inches="tight")
    print(f"saved {path.relative_to(ROOT)}")

In [ ]:
def load_titles():
    if SOURCE == "db":
        import sqlite3
        with sqlite3.connect(DB_PATH) as con:
            titles = pd.read_sql("SELECT * FROM titles", con)
            genres = pd.read_sql(
                "SELECT tg.title_id, g.genre_name AS genre "
                "FROM title_genres tg JOIN genres g USING (genre_id)", con)
        return titles, genres
    df = pd.read_csv(TMDB_CSV)
    genres = (df.assign(genre=df["genres"].str.split("|"))
                .explode("genre").dropna(subset=["genre"])
                [["id", "media_type", "release_year", "genre"]])
    return df, genres


titles, title_genres = load_titles()
print(f"{len(titles):,} titles | {title_genres['genre'].nunique()} genres")
titles.head()

## Chart 1: Movies vs TV series

**Important for the report:** the TMDB extract requests the same number of pages for movies and TV, so the TMDB sample is roughly 50/50 *by design*. That says nothing about Netflix itself. For the real catalog split we use the full Kaggle catalog (~8,800 titles), and show the TMDB sample next to it so the difference is explained, not hidden.

In [ ]:
kaggle = pd.read_csv(KAGGLE_CSV)
real_split = kaggle["type"].map({"Movie": "Movies", "TV Show": "TV series"}).value_counts()
sample_split = titles["media_type"].map({"movie": "Movies", "tv": "TV series"}).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
for ax, counts, label in [(axes[0], real_split, "Full Netflix catalog (Kaggle)"),
                          (axes[1], sample_split, "TMDB sample used in this project")]:
    counts = counts.reindex(["Movies", "TV series"]).fillna(0)
    wedges, _, autotexts = ax.pie(
        counts, labels=None, colors=PALETTE[:2], startangle=90, counterclock=False,
        autopct=lambda p: f"{p:.0f}%", pctdistance=0.78,
        wedgeprops={"width": 0.42, "edgecolor": "white", "linewidth": 2})
    for t in autotexts:
        t.set_color("white"); t.set_fontweight("bold"); t.set_fontsize(12)
    ax.text(0, 0, f"{int(counts.sum()):,}\ntitles", ha="center", va="center",
            fontsize=13, fontweight="bold")
    ax.set_title(label, fontsize=12)

fig.legend(wedges, ["Movies", "TV series"], loc="lower center", ncol=2, frameon=False)
fig.suptitle("Movies vs TV series on Netflix", fontsize=15, fontweight="bold")
add_source(fig, "Sources: Kaggle 'Netflix Movies and TV Shows'; TMDB API")
fig.tight_layout(rect=(0, 0.06, 1, 0.95))
save(fig, "01_movies_vs_tv_donut.png")
plt.show()

print((real_split / real_split.sum() * 100).round(1).to_string())

**Insight (write 1–2 sentences after running):**

_e.g. "Movies make up about X% of the full Netflix catalog, roughly N times the number of TV series. The TMDB sample is balanced by design, so composition findings use the full catalog."_

## Chart 2: Top 15 genres by title count

In [ ]:
top = (title_genres.groupby(["genre", "media_type"]).size()
                  .unstack(fill_value=0)
                  .rename(columns={"movie": "Movies", "tv": "TV series"}))
top["Total"] = top.sum(axis=1)
top = top.sort_values("Total", ascending=False).head(15).iloc[::-1]

fig, ax = plt.subplots(figsize=(10, 7))
cols = [c for c in ["Movies", "TV series"] if c in top.columns]
top[cols].plot.barh(stacked=True, ax=ax, color=PALETTE[:len(cols)], width=0.75)
for i, total in enumerate(top["Total"]):
    ax.text(total + top["Total"].max() * 0.01, i, f"{total:,}", va="center", fontsize=9)

ax.set_title("Top 15 genres by number of titles")
ax.set_xlabel("Number of titles (a title can have several genres)")
ax.set_ylabel("")
ax.legend(title="", loc="lower right", frameon=False)
ax.set_xlim(0, top["Total"].max() * 1.1)
add_source(fig)
fig.tight_layout()
save(fig, "02_top_genres_bar.png")
plt.show()

share = (top["Total"].iloc[::-1] / titles.shape[0] * 100).round(1)
print("% of titles tagged with each genre:")
print(share.head(5).to_string())

**Insight (write 1–2 sentences after running):**

_e.g. "Drama is the most common genre, tagged on X% of titles, followed by Y and Z. Animation and Documentary appear far more often in TV series than in movies."_

## Chart 8: Genre evolution over time

Shares are used instead of raw counts, because newer years have far more titles and raw counts would only show that growth. Years are grouped into 5-year periods so each period has enough titles to be meaningful.

In [ ]:
TOP_N = 6
START_YEAR = 1990

g = title_genres.dropna(subset=["release_year"]).copy()
g["release_year"] = g["release_year"].astype(int)
g = g[g["release_year"] >= START_YEAR]
g["period"] = (g["release_year"] // 5) * 5

main = g["genre"].value_counts().head(TOP_N).index
g["genre_group"] = g["genre"].where(g["genre"].isin(main), "Other")

counts = g.groupby(["period", "genre_group"]).size().unstack(fill_value=0)
counts = counts[[*main, "Other"]]
min_titles = 20  # drop periods too small to give stable shares
counts = counts[counts.sum(axis=1) >= min_titles]
shares = counts.div(counts.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(11, 6))
colors = PALETTE[:TOP_N] + ["#C7C7CC"]
ax.stackplot(shares.index, shares.T.values, labels=shares.columns, colors=colors, alpha=0.92)
ax.set_xticks(shares.index)
last_year = int(g["release_year"].max())
ax.set_xticklabels([f"{p}–{str(min(p + 4, last_year))[2:]}" for p in shares.index], rotation=0)
ax.set_ylim(0, 100)
ax.set_xlim(shares.index.min(), shares.index.max())
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0f}%")
ax.set_title(f"How the genre mix has changed since {START_YEAR}")
ax.set_xlabel("Release period")
ax.set_ylabel("Share of genre tags")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, title="Genre")
add_source(fig)
fig.tight_layout()
save(fig, "08_genre_evolution_area.png")
plt.show()

change = (shares.iloc[-1] - shares.iloc[0]).round(1).sort_values()
print(f"Change in share, {shares.index[0]} → {shares.index[-1]} (percentage points):")
print(change.to_string())
print("\nTitles per period:", counts.sum(axis=1).to_dict())

**Insight (write 1–2 sentences after running):**

_e.g. "Since the early 2000s, X's share has grown by about N points while Y has declined. Note that TMDB's popularity sorting favours recent titles, so older periods have fewer titles."_

## Notes for the report (methodology)

- The genre charts count **genre tags**, not titles: a title tagged "Drama|Crime" counts once in each.
- TMDB uses partly different genre names for movies and TV (e.g. "Sci-Fi & Fantasy" for TV, "Science Fiction" for movies). They are kept as TMDB provides them; Person 2's transform step may merge them.
- The TMDB extract is sorted by popularity, so it over-represents recent, popular titles. Composition (chart 1) therefore uses the full Kaggle catalog.